# Prerequisites

In [2]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Auto-reload module eksternal setiap cell dijalankan ulang
%load_ext autoreload
%autoreload 2

# Load file artikel

In [16]:
from pathlib import Path
import pandas as pd

article_path = Path("article/test.md")
article = article_path.read_text(encoding="utf-8")

print(f"Loaded: {article_path} ({len(article)} karakter)")
print(article)  # preview

Loaded: article/test.md (6555 karakter)
# Nature and outdoors can help boost your health - here's how

## How 20 minutes of nature can boost your health

![](https://static.files.bbci.co.uk/bbcdotcom/web/20260427-074339-4d487e3684-web-3.2.0-4/grey-placeholder.png)!Getty Images A wide, front view angle shot of a family and their dog walking through a woodland forest in Northumberland, Northeastern England during the Covid-19 pandemic.Getty Images

Spending just 20 minutes in nature can trigger measurable changes inside your body

If you've ever felt calmer after a walk in the park or a stroll through the woods, it's not your imagination - it's biology.

Being outdoors can trigger measurable changes inside your body from lowering stress hormones, easing blood pressure and even improving your gut health.

You don't have to hike for hours to feel these benefits as maximum impact happens after just 20 minutes, so even a lunchtime walk to the park and a sandwich on a bench a few times a week

# Proses

## Pecah menjadi beberapa paragraph

In [17]:
import importlib
import preprocess

# Pastikan module eksternal memakai versi terbaru dari file .py
importlib.reload(preprocess)

# ambil query dari heading pertama markdown
first_nonempty_line = next((line.strip() for line in article.splitlines() if line.strip()), "")
# print(f"First non-empty line: {first_nonempty_line}")
query = first_nonempty_line.lstrip("#").strip()
print(f"Query: {query}")
query_tokens = preprocess.preProcess(query)
print(f"Query Tokens: {query_tokens}\n")

paragraphs = preprocess.splitParagraph(article)
print(f"Split into {len(paragraphs)} paragraphs")


Query: Nature and outdoors can help boost your health - here's how
Query Tokens: ['natur', 'outdoor', 'help', 'boost', 'health']

Split into 7 paragraphs


[nltk_data] Downloading package punkt to /home/juana/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## Proses setiap Paragraph

In [18]:
import preprocess
import counter

# ===============================
# STORAGE
# ===============================
tf_global = {}          # TF semua dokumen
tf_paragraph = {}       # TF per paragraph
df_all = {}             # DF per paragraph
idf_paragraph = {}      # IDF per paragraph
tfidf_paragraph = {}    # TF-IDF per paragraph


doc_counter = 1

# ===============================
# MAIN LOOP
# ===============================
for i, (title, content) in enumerate(paragraphs.items(), start=1):
    
    paragraph_key = f"P{i}"
    print(f"Processing {paragraph_key}/{len(paragraphs)} : {title}")
    
    docs = preprocess.splitDocument(content)
    print(f"  Split into {len(docs)} documents")
    # -------------------------------
    # PREPROCESS ALL DOCS
    # -------------------------------
    clean_docs = []
    paragraph_tf = {}


    
    # hitung tf dan idf query
    doc_key = "QUERY"
    tf_entry = {
        term: counter.tf_counter(term, query_tokens)
        for term in query_tokens
    }
    tf_global[doc_key] = tf_entry
    paragraph_tf[doc_key] = tf_entry
        
        
    for doc in docs:
        clean_doc = preprocess.preProcess(doc)
        clean_docs.append(clean_doc)

        doc_key = f"D{doc_counter}"

        # Hitung TF
        tf_entry = {
            term: counter.tf_counter(term, clean_doc)
            for term in query_tokens
        }

        tf_global[doc_key] = tf_entry
        paragraph_tf[doc_key] = tf_entry

        doc_counter += 1

    # -------------------------------
    # HITUNG DF & IDF
    # -------------------------------
    N = len(clean_docs)

    df_terms = {}
    idf_terms = {}

    for term in query_tokens:
        df_value = counter.calc_df(term, clean_docs)
        df_terms[term] = df_value

        idf_terms[term] = (
            counter.calc_idf(N, df_value)
            if df_value > 0 else 0
        )

    # -------------------------------
    # SIMPAN HASIL
    # -------------------------------
    tf_paragraph[paragraph_key] = paragraph_tf
    df_all[paragraph_key] = df_terms
    idf_paragraph[paragraph_key] = idf_terms

    # -------------------------------
    # HITUNG TF-IDF
    # -------------------------------
    tfidf_paragraph[paragraph_key] = {
        doc_key: {
            term: paragraph_tf[doc_key][term] * idf_terms[term]
            for term in query_tokens
        }
        for doc_key in paragraph_tf
    }
    


Processing P1/7 : 1
  Split into 0 documents
Processing P2/7 : 2
  Split into 6 documents
Processing P3/7 : 3
  Split into 5 documents
Processing P4/7 : 4
  Split into 7 documents
Processing P5/7 : 5
  Split into 4 documents
Processing P6/7 : 6
  Split into 6 documents
Processing P7/7 : 7
  Split into 8 documents


In [19]:
# print(tf_paragraph)
# print(tfidf_paragraph)
# print(df_all)
for para_key, docs in tfidf_paragraph.items():
    print(f"\n{'='*40}")
    print(f"  {para_key}")
    print(f"{'='*40}")
    
    print("\n[TF]")
    display(pd.DataFrame(tf_paragraph[para_key]).round(3))
    
    print("\n[DF]")
    display(pd.Series(df_all[para_key]).round(3).to_frame("df"))
    
    # print("\n[IDF]")
    # display(pd.Series(idf_paragraph[para_key]).round(3).to_frame("idf"))
    
    print("\n[TF-IDF]")
    display(pd.DataFrame(tfidf_paragraph[para_key]).round(3))
    
    # print("\n[VSM Cosine Similarity]")
    # para_vsm = {dk: vsm_scores[dk] for dk in docs}
    # display(pd.Series(para_vsm).round(3).to_frame("score"))


  P1

[TF]


,QUERY
natur,1
outdoor,1
help,1
boost,1
health,1



[DF]


,df
natur,0
outdoor,0
help,0
boost,0
health,0



[TF-IDF]


...

,QUERY,D29,D30,D31,D32,D33,D34,D35,D36
natur,1,1,0,1,0,0,0,1,0
outdoor,1,0,0,0,0,0,0,0,0
help,1,1,0,0,0,1,1,0,1
boost,1,0,0,0,0,0,0,0,0
health,1,0,0,0,0,0,0,0,0



[DF]


,df
natur,3
outdoor,0
help,4
boost,0
health,0



[TF-IDF]


,QUERY,D29,D30,D31,D32,D33,D34,D35,D36
natur,1.981,1.981,0.0,1.981,0.0,0.000,0.000,1.981,0.000
outdoor,0.000,0.000,0.0,0.000,0.0,0.000,0.000,0.000,0.000
help,1.693,1.693,0.0,0.000,0.0,1.693,1.693,0.000,1.693
boost,0.000,0.000,0.0,0.000,0.0,0.000,0.000,0.000,0.000
health,0.000,0.000,0.0,0.000,0.0,0.000,0.000,0.000,0.000


## hitung jarak dokumen dan query

akar sigma W ** 2 

In [20]:
jarak = {}

for para_key, docs in tfidf_paragraph.items():
    jarak_per_paragraph = {}
    print(f"\n{'='*40}")
    print(f"  {para_key}")
    print(f"{'='*40}")
    
    # print("\n[TF-IDF]")
    # display(pd.DataFrame(tfidf_paragraph[para_key]).round(3))
    # print("akar sum kuadrat")
    for doc_key, W in tfidf_paragraph[para_key].items():
        sum_of_squares = sum(value ** 2 for value in W.values())
        magnitude = sum_of_squares ** 0.5
        
        print(f"{doc_key}: {magnitude:.3f}")
        jarak_per_paragraph[doc_key] = magnitude
    
    jarak[para_key] = jarak_per_paragraph


  P1
QUERY: 0.000

  P2
QUERY: 4.583
D1: 2.099
D2: 0.000
D3: 3.493
D4: 0.000
D5: 3.635
D6: 2.099

  P3
QUERY: 3.237
D7: 0.000
D8: 0.000
D9: 0.000
D10: 1.916
D11: 5.560

  P4
QUERY: 3.235
D12: 0.000
D13: 2.946
D14: 1.336
D15: 2.673
D16: 1.336
D17: 1.336
D18: 1.336

  P5
QUERY: 1.693
D19: 1.693
D20: 0.000
D21: 0.000
D22: 1.693

  P6
QUERY: 4.412
D23: 1.693
D24: 3.417
D25: 0.000
D26: 3.881
D27: 2.099
D28: 0.000

  P7
QUERY: 2.606
D29: 2.606
D30: 0.000
D31: 1.981
D32: 0.000
D33: 1.693
D34: 1.693
D35: 1.981
D36: 1.693


In [21]:
for para_key, docs in jarak.items():
    print(f"\n{'='*40}")
    print(f"  {para_key}")
    print(f"{'='*40}")
    
    print("\n[Jarak]")
    display(pd.Series(docs).round(3).to_frame("jarak").T)


  P1

[Jarak]


,QUERY
jarak,0.0



  P2

[Jarak]


,QUERY,D1,D2,D3,D4,D5,D6
jarak,4.583,2.099,0.0,3.493,0.0,3.635,2.099



  P3

[Jarak]


...

,QUERY,D19,D20,D21,D22
jarak,1.693,1.693,0.0,0.0,1.693



  P6

[Jarak]


,QUERY,D23,D24,D25,D26,D27,D28
jarak,4.412,1.693,3.417,0.0,3.881,2.099,0.0



  P7

[Jarak]


,QUERY,D29,D30,D31,D32,D33,D34,D35,D36
jarak,2.606,2.606,0.0,1.981,0.0,1.693,1.693,1.981,1.693


## hitung dot product


In [22]:
dot_product = {}
for para_key, docs in tfidf_paragraph.items():
    # print(f"\n{'='*40}")
    # print(f"  {para_key}")
    # print(f"{'='*40}")
    # display(pd.DataFrame(tfidf_paragraph[para_key]).round(3))
    dot_product_paragraph = {}
    for doc_key, W in tfidf_paragraph[para_key].items():
        # print(f"Document: {doc_key}")
        # print(f"TF-IDF Vector: {W}")
        Q = tfidf_paragraph[para_key].get("QUERY", {})
        # print(f"Query TF-IDF Vector: {Q}")
        dot_product_value = sum(W.get(term, 0) * Q.get(term, 0) for term in Q)
        dot_product_paragraph[doc_key] = dot_product_value
        
    dot_product[para_key] = dot_product_paragraph
    
print(dot_product)

{'P1': {'QUERY': 0}, 'P2': {'QUERY': 21.00644154847092, 'D1': 4.404173538148803, 'D2': 0.0, 'D3': 12.198094472173313, 'D4': 0.0, 'D5': 13.212520614446408, 'D6': 4.404173538148803}, 'P3': {'QUERY': 10.48133638791522, 'D7': 0.0, 'D8': 0.0, 'D9': 0.0, 'D10': 3.672170169066785, 'D11': 17.29050260676366}, 'P4': {'QUERY': 10.464544645566406, 'D12': 0.0, 'D13': 8.678386606307098, 'D14': 1.7861580392593075, 'D15': 3.572316078518615, 'D16': 1.7861580392593075, 'D17': 1.7861580392593075, 'D18': 1.7861580392593075}, 'P5': {'QUERY': 2.8667473750380923, 'D19': 2.8667473750380923, 'D20': 0.0, 'D21': 0.0, 'D22': 2.8667473750380923}, 'P6': {'QUERY': 19.46901538536021, 'D23': 2.8667473750380923, 'D24': 11.6750944513357, 'D25': 0.0, 'D26': 15.064841847211406, 'D27': 4.404173538148803, 'D28': 0.0}, 'P7': {'QUERY': 6.790431904625086, 'D29': 6.790431904625086, 'D30': 0.0, 'D31': 3.923684529586993, 'D32': 0.0, 'D33': 2.8667473750380923, 'D34': 2.8667473750380923, 'D35': 3.923684529586993, 'D36': 2.866747375

In [23]:
cosine_similarity = {}
print("\n[dot product]")
for para_key, docs in dot_product.items():
    cosine_similarity_per_paragraph = {}
    print(f"\n{'='*40}")
    print(f"  {para_key}")
    print(f"{'='*40}")
    
    # print(docs)
    for term in docs:
        
        similarity = counter.calc_cosine_similarity(docs[term], jarak[para_key].get(term, 0.0), jarak[para_key].get("QUERY", 0.0))
        # print(f"{similarity:.3f}")
        cosine_similarity_per_paragraph[term] = similarity

    cosine_similarity[para_key] = cosine_similarity_per_paragraph

    display(pd.Series(cosine_similarity_per_paragraph).round(3).to_frame("cosine_similarity").T)


[dot product]

  P1


,QUERY
cosine_similarity,0.0



  P2


,QUERY,D1,D2,D3,D4,D5,D6
cosine_similarity,1.0,0.458,0.0,0.762,0.0,0.793,0.458



  P3


...

,QUERY,D19,D20,D21,D22
cosine_similarity,1.0,1.0,0.0,0.0,1.0



  P6


,QUERY,D23,D24,D25,D26,D27,D28
cosine_similarity,1.0,0.384,0.774,0.0,0.88,0.476,0.0



  P7


,QUERY,D29,D30,D31,D32,D33,D34,D35,D36
cosine_similarity,1.0,1.0,0.0,0.76,0.0,0.65,0.65,0.76,0.65


# ranking dokumen

In [24]:
n = 3
top_n_results = {}  # Wadah untuk menyimpan hasil akhir

for para_key, docs in jarak.items():
    print(f"\n{'='*40}")
    print(f"  {para_key}")
    print(f"{'='*40}")
    
    # Hapus QUERY agar tidak ikut terhitung
    cosine_similarity[para_key].pop("QUERY", None)
    
    # Urutkan berdasarkan skor tertinggi
    rank = dict(sorted(cosine_similarity[para_key].items(), key=lambda item: item[1], reverse=True))
    
    # Ambil n teratas menggunakan slicing list dan ubah kembali ke dict
    top_n = dict(list(rank.items())[:n])
    top_n_results[para_key] = top_n # Simpan ke variabel luar
    
    print(f"\n[Top {n} Rank]")
    for term, score in top_n.items():
        print(f"  {term}: {score:.3f}")

# Sekarang variabel 'top_n_results' berisi dokumen terbaik untuk setiap para_key



  P1

[Top 3 Rank]

  P2

[Top 3 Rank]
  D5: 0.793
  D3: 0.762
  D1: 0.458

  P3

[Top 3 Rank]
  D11: 0.961
  D10: 0.592
  D7: 0.000

  P4

[Top 3 Rank]
  D13: 0.911
  D14: 0.413
  D15: 0.413

  P5

[Top 3 Rank]
  D19: 1.000
  D22: 1.000
  D20: 0.000

  P6

[Top 3 Rank]
  D26: 0.880
  D24: 0.774
  D27: 0.476

  P7

[Top 3 Rank]
  D29: 1.000
  D31: 0.760
  D35: 0.760


## Summary

In [ ]:
doc_text_map = {}
doc_counter = 1

for _, content in paragraphs.items():
    docs = preprocess.splitDocument(content)
    for doc in docs:
        doc_key = f"D{doc_counter}"
        doc_text_map[doc_key] = doc.strip()
        doc_counter += 1

# Buat summary dari top_n_results
summary_per_paragraph = {}
summary_lines = []

for para_key, ranked_docs in top_n_results.items():
    selected_docs = []
    
    # ranked_docs sudah berisi 2 skor terbesar per paragraph
    for doc_key, score in ranked_docs.items():
        original_text = doc_text_map.get(doc_key, "")
        if original_text:
            selected_docs.append({
                "doc_key": doc_key,
                "score": score,
                "text": original_text
            })
    
    summary_per_paragraph[para_key] = selected_docs
    
    # Versi ringkas: hanya teks kalimat asli
    for item in selected_docs:
        summary_lines.append(f"{item['text']}")
    summary_lines.append("")  # Baris kosong sebagai pemisah

# Gabungkan semua menjadi summary akhir
final_summary = "\n".join(summary_lines).strip()

print(f"\n=== SUMMARY (Top-{n} per Paragraph) ===\n")
print(final_summary)

# Optional: Simpan summary ke file
output_file = f"summary/summary_{query.replace(' ', '_')}.txt"
with open(output_file, "w", encoding="utf-8") as f:
    f.write(final_summary)
print(f"\n✓ Summary telah disimpan ke '{output_file}'")


=== SUMMARY (Top-3 per Paragraph) ===

Here are four ways that being among nature can help improve your health.
Being outdoors can trigger measurable changes inside your body from lowering stress hormones, easing blood pressure and even improving your gut health.
Spending just 20 minutes in nature can trigger measurable changes inside your body

The evidence for the benefits of spending time in nature is compelling enough that some areas have trialled so called green social prescribing connecting people with nature to improve their phsyical and mental health, with a positive impact on happiness and wellbeing.
A UK study, involving nearly 20,000 people, found that those who spent at least a total of 120 minutes every week in greenery were significantly more likely to report good health and higher psychological well-being.
When you see green trees, smell pine and hear gentle rustling leaves or the sound of birdsong, your autonomic nervous system - a network of nerves controlling unconsc